# Module 1 — AI Systems Thinking & Architecture Decisions

**Google Colab deep-practice notebook.** Predict → Build → Break → Measure → Explain → Defend.

Core ladder: deterministic software → LLM app → RAG → tool-using agent → stateful agent → multi-agent.


## Objectives + concept map
By the end you can: (1) translate requirements into architecture, (2) identify the minimum sufficient control plane, (3) quantify quality/latency/cost/risk trade-offs, (4) reject unnecessary agent complexity.

**Concept map:** requirements → architecture pattern → control boundaries → failure modes → KPIs → ADR → production decision.


## Theory: mechanism before framework
An AI system is not defined by whether it uses an agent framework. Start from the contract: what must be generated, what knowledge is needed, what actions are allowed, what state survives, and what must be verified. Add complexity only when it buys measurable capability.


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Requirements:
    generation: bool
    knowledge: bool
    actions: bool
    roles: int = 1
    risk: str = 'low'
    human_approval: bool = False

def recommend(r):
    if not r.generation: return 'deterministic_software'
    if not r.knowledge and not r.actions: return 'llm_application'
    if r.actions: return 'evaluate_multi_agent' if r.roles > 3 and r.risk != 'high' else 'agent_or_workflow'
    return 'rag'


## BUILD — guided implementation
Run the classifier below, then change exactly one requirement at a time. Predict how the architecture should change before executing.


In [ ]:
cases = [
 Requirements(False,False,False),
 Requirements(True,False,False),
 Requirements(True,True,False),
 Requirements(True,True,True,risk='high',human_approval=True),
 Requirements(True,True,True,roles=5),
]
for c in cases: print(c, '=>', recommend(c))


## TRY — architecture triage
Classify: email formatter; changing HR policy assistant; order-status API assistant; security investigation assistant; complex research with specialists. For each, record minimum architecture, rejected alternative, biggest risk, and primary KPI.


In [ ]:
scenarios = {
 'email_formatter': Requirements(True,False,False),
 'policy_assistant': Requirements(True,True,False),
 'order_status': Requirements(True,True,True,risk='medium'),
 'security_investigation': Requirements(True,True,True,risk='high',human_approval=True),
 'research_team': Requirements(True,True,True,roles=5),
}
for name, req in scenarios.items(): print(f'{name:24} -> {recommend(req)}')


## Industry scenario — enterprise banking
A support assistant must answer policy questions from governed documents, never cross customer tenants, and may prepare but not independently execute high-impact account actions. **Design the minimum architecture.** Identify the trust boundary, approval boundary, evidence requirement, and release-blocking KPI.


## MEASURE — KPI scorecard
Use task success, retrieval quality, groundedness, p95 latency, reliability, cost/task, safety violations, and trace coverage. A metric becomes a release blocker when violating it creates unacceptable customer, security, financial, or regulatory risk.


In [ ]:
kpis = ['task_success','retrieval','groundedness','p95_latency_ms','error_rate','cost_per_task','safety_violations','trace_coverage']
print('KPI scorecard fields:', kpis)
sample = {'task_success':0.94,'groundedness':0.98,'p95_latency_ms':1800,'error_rate':0.01,'cost_per_task':0.04,'safety_violations':0,'trace_coverage':1.0}
print(sample)


## BREAK — intentionally unsafe architecture
Bad design: the model can directly authorize refunds, retrieved documents are treated as trusted instructions, tenant filters are omitted, and the loop has no hard budget.

**TODO:** list the failure for each defect, then add deterministic controls. Expected controls: tool allowlist + authorization, untrusted-content labeling, tenant enforcement outside the model, bounded loop/budget, verification and audit.


In [ ]:
def control_for(defect):
    controls = {
      'refund': 'authorization + exact approval binding + audit',
      'untrusted_docs': 'treat retrieval as data, never policy',
      'tenant_leak': 'server-side tenant/ACL filter before retrieval/tool use',
      'unbounded_loop': 'hard step/time/token/tool-call budget + progress check'
    }
    return controls[defect]
for d in ['refund','untrusted_docs','tenant_leak','unbounded_loop']:
    print(d, '=>', control_for(d))


## Debugging challenge
If a system passes average quality but has rare unauthorized actions, do **not** optimize the average. Locate the violated invariant, reproduce it with a minimal trace, add a regression case, and make the security gate block release.


## Reference solution / reasoning
A strong answer chooses the simplest architecture that satisfies requirements; puts authorization, tenant isolation, budgets and verification in deterministic code; treats model/retrieval/tool outputs as untrusted; measures quality and operational cost separately; and records the decision in an ADR. Adding agents without a measurable capability gain is architectural overreach.


## Gold system-design challenge
Produce an ADR for an enterprise assistant serving 50M documents and read-only APIs. Defend every component using: capability gained, failure prevented, measurable KPI, operational cost, and rollback strategy.

### Mastery gate
Pass only if you can explain why a simpler architecture is insufficient, identify failure modes, select release-blocking KPIs, design deterministic safety boundaries, and defend the decision without saying 'because an agent/RAG framework is standard'.
